<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training%20/4_1_model_training_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_1_model_random_forest

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [14]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


In [15]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [16]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [17]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [18]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [19]:
mnq_train = load_data("train")
mnq_valid = load_data("valid")
mnq_test = load_data("test")

### 1.2. Información de datasets


In [20]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [21]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(197, np.float64(301.0))

### 1.3. Carga de listado de features por ventana de tiempo

In [22]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [23]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [24]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [25]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

### 2.1 Carga de ventanas 30 minutos

In [26]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [27]:
xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [28]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [29]:
xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [30]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [31]:
xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Entrenamiento de Modelo

### 3.0. Funciones

#### Instalación e importación de librerías especificas

In [2]:
!pip install optuna --quiet

In [3]:
!pip uninstall -y scikit-learn scikit-image threadpoolctl joblib --quiet
!pip install --no-cache-dir "scikit-learn==1.5.1" "optuna==3.6.1" "numpy>=1.24,<3" "scipy>=1.10" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 227.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.


In [4]:
import numpy as np
import optuna
import sklearn, optuna, numpy, scipy
import sklearn.ensemble as sken
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [5]:
print("sklearn:", sklearn.__version__)
print("optuna:", optuna.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)

sklearn: 1.5.1
optuna: 3.6.1
numpy: 2.0.2
scipy: 1.16.2


#### Función para evaluación de modelo


In [37]:
# --- función de evaluación ---
def evaluate_model(model, X, y, preds):
    rmse = mean_squared_error(y, preds, squared=False)
    mae = mean_absolute_error(y, preds)
    r2 = r2_score(y, preds)
    mape = np.mean(np.abs((y - preds) / (y + 1e-8))) * 100  # evitar división por cero
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}

## 4. Función para tuneo de hiperparametros


Proceso de optimización:

1. Optuna llama a objective con un conjunto de hiperparámetros.
2. Entrena un modelo con esos hiperparámetros.
3. Evalúa el modelo en validación.
4. Devuelve el RMSE, que Optuna usa para guiar la búsqueda.

### 4.0. Funciones

In [36]:
# --- función de objetivo para Optuna ---
def objective(trial, X_train, y_train, X_valid, y_valid):

    #Definición de espacio de búsqueda de hiperparámetros
    #Cada trial.suggest_* le decimos a Optuna: prueba un valor de este hiperparámetro dentro de este rango/espacio.
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "n_jobs": -1,
        "random_state": 42
    }

    #Entrenamiento del modelo con los parametros del trial
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)

    #Predicción y evaluación
    ## El modelo predice sobre el conjunto de validación
    ## Se calculan las métricas (RMSE, MAE, R², MAPE) con la función evaluate_model.
    preds = model.predict(X_valid)
    metrics = evaluate_model(model, X_valid, y_valid, preds)


    # Devolvemos la métrica que Optuna va a optimizar, elegimos el RMSE (minimizarlo).
    # Optuna elige los hiperparámetros que produzcan el RMSE más bajo en validación.
    return metrics["RMSE"]

In [38]:
def tuning_hiperparam(X_train, y_train, X_valid, y_valid, n_trials=30):

    #n_trials: cantidad de pruebas (intentos) que Optuna va a hacer buscando la mejor combinación de hiperparámetros.

    # Creamos un estudio de Optuna, e intentamos minimizar la métrica elegida.
    study = optuna.create_study(direction="minimize")
    # Se ejecuta la búsqueda de hiperparámetros
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_valid, y_valid), n_trials=n_trials)

    # Recuperamos los mejores hiperparámetros y los guardamos en best_params
    best_params = study.best_params

    return best_params


### 4.1. Hiperparámetros para 30 minutos.

In [ ]:
best_params_30 = tuning_hiperparam(X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, n_trials=30)

[I 2025-10-01 20:44:18,720] A new study created in memory with name: no-name-d0943159-4367-4260-9de9-4e0e314589ed


In [ ]:
print("Mejores hiperparámetros:", best_params_30)

### 4.2. Hiperparámetros para 60 minutos.

In [ ]:
best_params_60 = tuning_hiperparam(X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, n_trials=30)

In [ ]:
print("Mejores hiperparámetros:", best_params_60)

### 4.3. Hiperparámetros para 90 minutos.

In [ ]:
best_params_90 = tuning_hiperparam(X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, n_trials=30)

In [ ]:
print("Mejores hiperparámetros:", best_params_90)

## 5. Entrenamiento de modelo

### 5.0. Funciones

In [ ]:
def train_model (best_params, X_train, y_train, X_valid, y_valid):

    # Entrenamos el model final con los mejores hiperparámetros
    # Se entrena un RandomForest definitivo usando todos los datos de entrenamiento y los best_params
    best_model = RandomForestRegressor(**best_params, n_jobs=-1, random_state=42)
    best_model.fit(X_train, y_train)

    # Evaluación en validación
    preds = best_model.predict(X_valid)
    # Se calculan las métricas de desempeño.

    return best_model, preds

### 5.1. Entrenamiento 30min

In [ ]:
best_model_30, preds_30 = train_model(best_params_30, X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30)

### 5.2. Entrenamiento 60min

In [ ]:
best_model_60, preds_60 = train_model(best_params_60, X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60)

### 5.3. Entrenamiento 90min

In [ ]:
best_model_90, preds_90 = train_model(best_params_90, X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90)

## 6.  Evaluación de modelo

### 6.1. Métricas de modelo 30min

In [ ]:
metrics_30 = evaluate_model(best_model_30, X_valid_30_scaled, y_valid_30, preds_30)
print("Métricas en validación:", metrics_30)

### 6.2. Métricas de modelo 60min

In [ ]:
metrics_60 = evaluate_model(best_model_60, X_valid_60_scaled, y_valid_60, preds_60)
print("Métricas en validación:", metrics_60)

### 6.3. Métricas de modelo 90min

In [ ]:
metrics_90 = evaluate_model(best_model_90, X_valid_90_scaled, y_valid_90, preds_90)
print("Métricas en validación:", metrics_90)